### Anchor Embeddings
Use: generate embeddings from h5 files generated because they are not present in SGA 2025. 
Instructions: change RUN_NAME to match that of the desired generated file

In [1]:
from ssl_legacysurvey.utils import plotting_tools as plt_tools
from ssl_legacysurvey.utils import load_data, format_logger
from ssl_legacysurvey.data_loaders import datamodules
from ssl_legacysurvey.moco.moco2_module import Moco_v2

from dataclasses import dataclass, field
from pathlib import Path
import time
import torch
import pickle
import numpy as np
from tqdm import tqdm
import glob

In [2]:
@dataclass
class Config:
    output_dir: Path = Path("/pscratch/sd/q/qshimp/Sorter")
    ssl_dir: Path = Path("/global/cfs/cdirs/desicollab/users/ioannis/SGA/2025/ssl")

    seed: int = 67
    checkpoint: Path = Path("/global/homes/q/qshimp/morphology_classification"
                            "/Binary-classifier/resnet50.ckpt")
    npix: int = 152
    embedding_batch_size: int = 128

    augmentations: str = "rrjc"
    jitter_lim: int = 0

# Change these
RUN_NAME = "SGA_2025"
VERBOSE = 2

config = Config()
anchor_h5_path = str("/pscratch/sd/q/qshimp/h5py-files/SGA2025.h5")

run_path = config.output_dir / "binary_classifier" / RUN_NAME
run_path.mkdir(parents=True, exist_ok=True)

In [3]:
def build_ssl_params(config):
    return {
        "augmentations": config.augmentations,
        "npix_out": config.npix,
        "jitter_lim": config.jitter_lim,
        "verbose": False,
    }

def read_directory(directory):
    files = glob.glob(f"{directory}/ssl-c*")
    return files
 
 
def generate_embeddings_streaming(
    dataset,
    backbone,
    config,
    output_path,
    verbose=1,
    checkpoint_every=5000
):
    """
    Process images directly from DecalsDataset and generate embeddings
    in batches. Progress is saved to a single HDF5 checkpoint so the run
    can resume after a crash.

    Pipeline:
        DecalsDataset
            -> transformed image
            -> ResNet encoder
            -> 2048-d embedding
            -> HDF5 checkpoint
    """

    import h5py

    output_path = Path(output_path)

    checkpoint_path = output_path.with_name(
        output_path.stem + "_checkpoint.h5"
    )

    if output_path.exists():
        print(f"Loaded completed embeddings from {output_path}")
        return torch.load(output_path, map_location="cpu")

    if len(dataset) == 0:
        raise ValueError("Cannot generate embeddings from empty dataset.")

    # ------------------------------------------------------------
    # Device
    # ------------------------------------------------------------

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print(f"Using device: {device}")

    backbone = backbone.to(device)
    backbone.eval()

    # ------------------------------------------------------------
    # Determine embedding dimension
    # ------------------------------------------------------------

    sample_image, _ = dataset[0]

    with torch.no_grad():
        sample_embedding = backbone(
            sample_image.unsqueeze(0).to(device)
        ).cpu()

    embedding_dim = sample_embedding.shape[1]

    print(f"Embedding dimension: {embedding_dim}")

    # ------------------------------------------------------------
    # Open/create checkpoint
    # ------------------------------------------------------------

    with h5py.File(checkpoint_path, "a") as f:

        if "embeddings" not in f:

            f.create_dataset(
                "embeddings",
                shape=(len(dataset), embedding_dim),
                dtype=np.float32,
                chunks=(checkpoint_every, embedding_dim),
            )

            f.attrs["processed"] = 0

        processed = int(f.attrs["processed"])

        print(
            f"Checkpoint contains "
            f"{processed}/{len(dataset)} processed galaxies."
        )

        # --------------------------------------------------------
        # Process in 5000-galaxy blocks
        # --------------------------------------------------------

        t_start = time.perf_counter()

        for block_start in range(
            processed,
            len(dataset),
            checkpoint_every
        ):

            block_end = min(
                block_start + checkpoint_every,
                len(dataset)
            )

            block_embeddings = []

            # ----------------------------------------------------
            # Process this block in model-sized batches
            # ----------------------------------------------------

            for start in tqdm(
                range(
                    block_start,
                    block_end,
                    config.embedding_batch_size
                ),
                desc=f"Embedding {block_start}-{block_end}",
                disable=(verbose < 2),
            ):

                end = min(
                    start + config.embedding_batch_size,
                    block_end
                )

                # Get transformed images directly from DecalsDataset
                batch_images = torch.stack([
                    dataset[i][0]
                    for i in range(start, end)
                ])

                batch_images = batch_images.to(device)

                with torch.no_grad():
                    batch_embeddings = backbone(
                        batch_images
                    ).cpu()

                block_embeddings.append(batch_embeddings)

            # ----------------------------------------------------
            # Combine this 5000-galaxy block
            # ----------------------------------------------------

            block_embeddings = torch.cat(
                block_embeddings,
                dim=0
            ).numpy()

            # ----------------------------------------------------
            # Save block to ONE checkpoint file
            # ----------------------------------------------------

            f["embeddings"][block_start:block_end] = (
                block_embeddings
            )

            # Only update progress after the data has been written
            f.attrs["processed"] = block_end

            # Force data to disk
            f.flush()

            elapsed = time.perf_counter() - t_start

            print(
                f"\nSaved embedding checkpoint: "
                f"{block_end}/{len(dataset)} "
                f"({100 * block_end / len(dataset):.1f}%)"
            )

            print(
                f"  Checkpoint: {checkpoint_path}"
            )

            print(
                f"  Time since start: {elapsed / 60:.1f} min"
            )

    # ------------------------------------------------------------
    # Entire dataset is complete
    # ------------------------------------------------------------

    print("\nEmbedding generation complete.")
    print("Creating final PyTorch embedding file...")

    with h5py.File(checkpoint_path, "r") as f:
        embeddings = torch.from_numpy(
            f["embeddings"][:]
        )

    torch.save(
        embeddings,
        output_path
    )

    print(
        f"Saved final embeddings to {output_path}"
    )

    return embeddings

In [4]:
# ---- Anchors: still need real image -> embedding generation ----    
print("Loading anchor galaxies...")

loader = load_data.DecalsDataLoader(
    image_dir=anchor_h5_path,
    npix_in=config.npix
)

anchor_gals = loader.get_data(
    -1,
    fields=["sgaid", "ra", "dec"],
    npix_out=config.npix
)

print(f"Loaded {len(anchor_gals['sgaid'])} anchor galaxies.")

anchor_indices = anchor_gals["inds"].astype(np.int64)

label_array = np.column_stack([
    anchor_indices,
    np.zeros(len(anchor_indices), dtype=np.int64)
])

label_path = run_path / "anchor_label_index.npy"
np.save(label_path, label_array)

print(f"Saved dataset index file to {label_path}")

model = Moco_v2.load_from_checkpoint(checkpoint_path=config.checkpoint)
backbone = model.encoder_q
backbone.fc = torch.nn.Identity()

params = build_ssl_params(config)
transform = datamodules.DecalsTransforms(config.augmentations, 
                                         {"npix_out": config.npix, "jitter_lim": config.jitter_lim})

dataset = datamodules.DecalsDataset(anchor_h5_path, str(label_path), transform, params)

anchor_embeddings = generate_embeddings_streaming(
    dataset,
    backbone,
    config,
    run_path / "anchor_embeddings.pt",
    VERBOSE,
    checkpoint_every=5000
)

bridge_path = run_path / "anchor_artifacts.pkl"
bridge_data = {
    "anchor_gals": anchor_gals,
    "anchor_embeddings": anchor_embeddings.detach().cpu().numpy(),
}
with open(bridge_path, "wb") as f:
    pickle.dump(bridge_data, f)
print(f"Saved anchor artifacts to {bridge_path}")

Loading anchor galaxies...
Loaded 445693 anchor galaxies.
Saved dataset index file to /pscratch/sd/q/qshimp/Sorter/binary_classifier/SGA_2025/anchor_label_index.npy
Using device: cuda
Embedding dimension: 2048
Checkpoint contains 0/445693 processed galaxies.


Embedding 0-5000: 100%|██████████| 40/40 [00:34<00:00,  1.16it/s]



Saved embedding checkpoint: 5000/445693 (1.1%)
  Checkpoint: /pscratch/sd/q/qshimp/Sorter/binary_classifier/SGA_2025/anchor_embeddings_checkpoint.h5
  Time since start: 0.6 min


Embedding 5000-10000: 100%|██████████| 40/40 [00:34<00:00,  1.16it/s]



Saved embedding checkpoint: 10000/445693 (2.2%)
  Checkpoint: /pscratch/sd/q/qshimp/Sorter/binary_classifier/SGA_2025/anchor_embeddings_checkpoint.h5
  Time since start: 1.2 min


...


Saved embedding checkpoint: 440000/445693 (98.7%)
  Checkpoint: /pscratch/sd/q/qshimp/Sorter/binary_classifier/SGA_2025/anchor_embeddings_checkpoint.h5
  Time since start: 54.7 min


Embedding 440000-445000: 100%|██████████| 40/40 [00:35<00:00,  1.13it/s]



Saved embedding checkpoint: 445000/445693 (99.8%)
  Checkpoint: /pscratch/sd/q/qshimp/Sorter/binary_classifier/SGA_2025/anchor_embeddings_checkpoint.h5
  Time since start: 55.3 min


Embedding 445000-445693: 100%|██████████| 6/6 [00:04<00:00,  1.41it/s]



Saved embedding checkpoint: 445693/445693 (100.0%)
  Checkpoint: /pscratch/sd/q/qshimp/Sorter/binary_classifier/SGA_2025/anchor_embeddings_checkpoint.h5
  Time since start: 55.4 min

Embedding generation complete.
Creating final PyTorch embedding file...
Saved final embeddings to /pscratch/sd/q/qshimp/Sorter/binary_classifier/SGA_2025/anchor_embeddings.pt
Saved anchor artifacts to /pscratch/sd/q/qshimp/Sorter/binary_classifier/SGA_2025/anchor_artifacts.pkl
